# Analyse des reports entre premier et deuxième tours des législatives 2024

*Salomé Cheysson*

Ce notebook est le rapport : chaque tableau et chaque graphe y est recalculé depuis
`data/reports_legislatives_2024.parquet` (régénérable avec
`uv run python -m reports_de_voix.estimer`). Après `uv sync`, ré-exécuter le
notebook suffit à tout rafraîchir.

Lecture des graphes de reports : un point par circonscription (trié du report le
plus faible au plus fort), le trait horizontal donne la plage de stabilité bootstrap
(95 % central) : quand elle est large, c'est elle, non le point, qui fait foi. Zoom
par molette ou glissement, indépendant par panneau.

In [1]:
import polars as pl

from reports_de_voix.donnees import PARQUET, SENSIBILITES, candidats
from reports_de_voix.graphiques import ABST, graphe_configurations, paires, panneaux
from reports_de_voix.tables import (
    configurations,
    matrice_exemple,
    mobilisation,
    r2_faible,
)

df = pl.read_parquet(PARQUET)
cand = candidats()

## Configurations de T2

D’après notre nuançage des candidats, on relève 19 types de configurations de second tour parmi les 497 circonscriptions analysées. Les 3 premières configurations concernent à elles seules 304 circonscriptions, et les 8 premières en couvrent 449.

In [2]:
configurations(df)

configuration,nombre,part
str,u32,f64
"""NFP / RN""",143,0.287726
"""ENS-HOR / RN""",103,0.207243
"""NFP / ENS-HOR / RN""",58,0.1167
"""RN""",38,0.076459
"""LR / RN""",37,0.074447
"""NFP / ENS-HOR""",27,0.054326
"""ENS-HOR / Union ext. droite""",23,0.046278
"""NFP / Union ext. droite""",20,0.040241
"""Autres""",48,0.096579


In [3]:
graphe_configurations(configurations(df))

alt.LayerChart(...)

## Matrices des reports

Une matrice de report prend la forme d’un tableau où chaque ligne correspond à une étiquette du premier tour, et chaque colonne correspond à une étiquette du second tour. La valeur à l’intersection d’une ligne et d’une colonne indique dans quel proportion les électeurs qui ont voté pour l’étiquette de la ligne au premier tour votent pour l’étiquette de la ligne au second tour.

Par exemple, pour la deuxième circonscription de la Somme, la matrice prend la forme suivante :

In [4]:
matrice_exemple(df)

T1 \ T2,NFP,RN,ENS-HOR
str,f64,f64,f64
"""NFP""",98.6,0.0,0.0
"""Gauche div.""",19.2,51.4,0.0
"""ENS-HOR""",0.0,2.2,94.7
"""Centre-droit""",27.4,9.8,43.5
"""LR""",2.0,12.4,82.0
"""Reconquête""",21.1,78.9,0.0
"""Union ext. droite""",62.5,37.5,0.0
"""RN""",0.0,100.0,0.0
"""abstention""",7.5,0.7,0.0


On y lit ainsi que 99 % des électeurs du NFP au 1er tour ont revoté NFP au 2ème tour, et que 79 % des électeurs qui ont choisi Reconquête au 1er tour ont voté pour le candidat RN au 2ème.

## Enseignements

### Front républicain et Front anti-FI

Les matrices de reports permettent de répondre à la question de la solidité du front républicain en regardant la façon dont les différents blocs se sont reportés dans les cas où le Rassemblement national était présent au 2ème tour. Les graphes ci-dessous distinguent les configurations de second tour (une configuration par ligne), et indique la façon dont les électeurs des trois principaux blocs « républicains » (en colonne, NFP, Ensemble et LR) se sont comportés au second tour.

### Duels NFP / RN — reports des électeurs ENS-HOR et LR

In [5]:
duels_nfp_rn = paires(df.filter(pl.col("configuration") == "NFP / RN"))
panneaux(duels_nfp_rn, cand, ["ENS-HOR → NFP", "LR → NFP", "ENS-HOR → RN", "LR → RN"])

alt.VConcatChart(...)

On observe dans ces duels une très grande variabilité des reports des électeurs d’Ensemble et Horizons vers les candidats NFP. Les meilleurs valeurs de reports obtenus, autour de 80 %, correspondent à des sortants (Delphine Batho, Yannick Monnet, Christine Pirès Beaune, Chantal Jourdan, Caroline Fiat…). Mais la plupart des reports sont inférieurs, autour de 55 %. Plusieurs candidats obtiennent des reports encore plus faibles, notamment un report très faible (environ 7 %) pour Raphaël Arnault, et des reports faibles pour Jérôme Legavre et Philippe Poutou.

Les reports des électeurs LR sont aussi très variables, mais nettement inférieurs à ceux vers le RN (médiane d’environ 22 % vers le NFP, contre 39 % vers le RN). Les très bons reports se font vers Ayda Hadizadeh, Sébastien Jumel et Jérôme Guedj.

### Duels ENS-HOR / RN — reports des électeurs NFP et LR

In [6]:
duels_ens_rn = paires(df.filter(pl.col("configuration") == "ENS-HOR / RN"))
panneaux(duels_ens_rn, cand, ["NFP → ENS-HOR", "LR → ENS-HOR", "NFP → RN", "LR → RN"])

alt.VConcatChart(...)

Les reports des électeurs du NFP vers le candidat Ensemble ou Horizons dans le cas d’un duel contre le RN se concentrent autour de 86 % en médiane et restent presque toujours supérieurs à 70 %.

Les reports LR sont généralement autour de 65 %, avec quelques valeurs extrêmes, qui semblent plus correspondre à des effets repoussoir, soit du candidat Ensemble, soit du candidat RN. Les bons reports LR vers le candidat Renaissance concernent les circos où celui-ci est opposé à José Gonzalez (implanté depuis longtemps) ou Anis Bouvard (qui a fait une prestation désastreuse à la télévision). Les plus mauvais reports concernent Marc Fesneau et Richard Ramos.

### Triangulaires NFP / ENS-HOR / RN — reports des électeurs LR

In [7]:
triangulaires = paires(df.filter(pl.col("configuration") == "NFP / ENS-HOR / RN"))
panneaux(triangulaires, cand, ["LR → NFP", "LR → ENS-HOR", "LR → RN"], cols=3)

alt.VConcatChart(...)

En cas de triangulaire NFP / Ensemble ou Horizons / RN, les reports LR se font généralement vers Ensemble, avec beaucoup de variabilité. Côté NFP, les reports LR sont au contraire très faibles : la médiane est proche de zéro et les meilleurs n’atteignent qu’environ un tiers des voix (Stéphane Hablot, Marianne Maximi). Côté RN, les bons reports se font vers les adversaires de Sébastien Ramage et Maximilien Jules-Arthur.

### Duels NFP / ENS-HOR — reports des électeurs LR et RN

In [8]:
duels_nfp_ens = paires(df.filter(pl.col("configuration") == "NFP / ENS-HOR"))
panneaux(duels_nfp_ens, cand, ["LR → NFP", "LR → ENS-HOR", "RN → NFP", "RN → ENS-HOR"])

alt.VConcatChart(...)

### Duels LR / RN — reports des électeurs NFP et ENS-HOR

In [9]:
duels_lr_rn = paires(df.filter(pl.col("configuration") == "LR / RN"))
panneaux(duels_lr_rn, cand, ["NFP → LR", "NFP → RN", "ENS-HOR → LR", "ENS-HOR → RN"])

alt.VConcatChart(...)

On constate ici aussi la bonne qualité des reports anti RN des électeurs NFP, qui se reportent eux aussi largement vers les candidats LR, quoique un peu moins que les électeurs Ensemble (médiane de 83 % contre 96 %).

### Reports vers les candidats NFP dissidents (Ruffin 80-01, Davi 13-05)

In [10]:
# Ruffin (80-01) et Davi (13-05) : NFP dissidents, nuancés « Gauche div. »
dissidents = df.filter(
    pl.col("circonscription").is_in(["80-01", "13-05"])
    & pl.col("source").is_in(["ENS-HOR", "LR"])
    & pl.col("destination").is_in(["NFP", "Gauche div."])
).with_columns(paire=pl.format("{} → NFP dissident", pl.col("source")))
panneaux(
    dissidents,
    cand,
    ["ENS-HOR → NFP dissident", "LR → NFP dissident"],
    cell_h=144,
    couleur="NFP",
)

alt.VConcatChart(...)

Les deux candidats NFP dissidents opposés au RN (Henrik Davi et François Ruffin) obtiennent des reports Ensemble dans la moyenne des candidats NFP.

### Reports vers le candidat NFP selon sa sensibilité (FI / PS / EELV / PCF)

In [11]:
sensibilites = pl.read_csv(SENSIBILITES)  # sensibilité du candidat NFP par circo
nfp_en_duel_rn = df.filter(
    (pl.col("configuration") == "NFP / RN")
    & (pl.col("destination") == "NFP")
    & pl.col("source").is_in(["ENS-HOR", "LR"])
).join(sensibilites, left_on="circonscription", right_on="circo", how="inner")
par_sensibilite = nfp_en_duel_rn.with_columns(
    paire=pl.format("{} · {}", pl.col("source"), pl.col("sensibilite"))
)
ordre = [f"{de} · {s}" for s in ["FI", "PS", "EELV", "PCF"] for de in ["ENS-HOR", "LR"]]
panneaux(par_sensibilite, cand, ordre, cell_h=258, couleur="NFP")

alt.VConcatChart(...)

Les reports des électeurs Ensemble et LR vers les candidats FI sont généralement inférieurs à ceux constatés vers les candidats des 3 autres sensibilités.

### La capacité à mobiliser les abstentionnistes du premier tour

In [12]:
mobilisation_abst = paires(df.filter(pl.col("source") == ABST))
panneaux(
    mobilisation_abst,
    cand,
    [f"{ABST} → {bloc}" for bloc in ["NFP", "ENS-HOR", "LR", "RN"]],
)

alt.VConcatChart(...)

Ce graphe indique la capacité des différents candidats à mobiliser les abstentionnistes du premier tour pour le deuxième tour. On constate que le centre de gravité des valeurs pour les candidats NFP est légèrement plus haut que pour les autres forces en présence.

Voici les plus hauts taux de report depuis l’abstention constatés :

In [13]:
mobilisation(df, cand)

circonscription,destination,candidat,report,mobilises,r2_echantillon
str,str,str,f64,f64,f64
"""69-06""","""NFP""","""AMARD Gabriel""",0.328795,11107.0,0.868394
"""69-01""","""NFP""","""BELOUASSA-CHERIFI Anaïs""",0.262819,5306.0,0.95651
"""75-15""","""NFP""","""VERZELETTI Céline""",0.244175,5963.0,0.830919
"""69-05""","""NFP""","""MATTEUCCI Fabrice""",0.228644,5444.0,0.718665
"""69-01""","""ENS-HOR""","""RUDIGOZ Thomas""",0.176921,3572.0,0.832972
…,…,…,…,…,…
"""92-09""","""ENS-HOR""","""SÉJOURNÉ Stéphane""",0.131163,2177.0,0.97746
"""75-03""","""NFP""","""BALAGE EL MARIKY Léa""",0.130396,2636.0,0.975442
"""95-10""","""NFP""","""TACHÉ Aurélien""",0.127319,3098.0,0.976834


## Annexe méthodologique

### Hypothèses et limites

L’hypothèse de départ de notre méthodologie est que le comportement des électeurs qui se sont prononcés pour un candidat est homogène sur le territoire de la circonscription. Dit autrement, les électeurs qui ont fait un choix A au 1er tour, se reporteront dans les mêmes proportions vers les différents choix de second tour, indépendament du lieu de vote au sein de la circonscription.

Cette méthodologie échouera donc dans les cas où les comportements de reports sont influencés par des caractéristiques locales plus fines que la circonscription. Un exemple fréquent correspond à un candidat qui serait maire d’une des communes de la circonscription : il attirerait potentiellement un électorat plus large que celui de son étiquette politique sur le territoire de sa commune, mais pas en dehors. Lors d’un second tour où il ne serait pas présent, les reports des électeurs qui ont voté pour lui ne seront donc pas identiques dans sa commune et en dehors.

### Calcul des matrices de reports

Une matrice de report donnée nous donne une prédiction du résultat du 2ème tour bureau de vote par bureau de vote. On peut ensuite évaluer la qualité de cette prédiction en comparant les scores prédits à ceux effectivement constatés. Les matrices de reports calculées ici correspondent aux matrices qui minimisent cet écart entre score prédit et score réel (au sens des moindres carrés).

Le but étant de donner un sens directement interprétable aux matrices de report, les matrices sont calculées sur la base du nombre de voix portées pour chaque candidat, au 1er et au 2ème tour, et non pas sur la base de la part des suffrages exprimés. Pour garder cette interprétabilité, on impose en conséquence que les taux de reports ne peuvent pas être négatifs, et que la somme des reports pour les électeurs qui se sont prononcés pour un candidat donné doit être inférieure à 1.

Cette méthodologie a l’avantage de nous permettre de calculer les taux de reports depuis et vers l’abstention de la même façon.

Comme il s’agit d’un problème linéaire, il est possible d’obtenir la solution optimale.

### Évaluation de la validité

Pour évaluer la validité de l’approche, on peut comparer la variance de l’écart entre second tour prédit et second tour réel, et la comparer à la variance totale constatée au second tour. L’indicateur obtenu s’appelle le $R^2$ qui mesure ainsi la part de la variance du second tour pris en compte dans notre prédiction. Un $R^2$ de $1$ correspond à la situation où l’écart entre valeur prédite et valeur réelle est partout nul.

Sur ce scrutin, l’ajustement en échantillon est presque parfait : le $R^2$ moyen y atteint $0.98$. Cette valeur est trompeuse, car le modèle estime beaucoup de coefficients à partir des seuls bureaux d’une circonscription et un tel ajustement est en grande partie mécanique ; il ne dit rien, à lui seul, de l’hypothèse d’homogénéité. Celle-ci se juge hors échantillon : on ajuste la matrice sur une partie des bureaux, puis on mesure sa prédiction sur les bureaux laissés de côté. Il vaut ici 0,975, à peine en deçà du 0,979 obtenu en échantillon : la relation agrégée se généralise bien d’un bureau à l’autre. C’est ce $R^2$ de validation croisée, et non celui calculé en échantillon, qui reflète la capacité prédictive réelle de la méthode.

La lecture des coefficients appelle la même prudence. Chaque taux de report n’est qu’une estimation, qu’on assortit d’une plage de stabilité obtenue en rééchantillonnant les bureaux par bootstrap (95 % central) ; comparer deux circonscriptions suppose de regarder ces plages, et non les seuls points. Nous parlons délibérément de plage de stabilité, et non d’intervalle de confiance : l’estimation étant contrainte de rester entre 0 et 100 %, le bootstrap par percentiles ne garantit pas, au voisinage de ces bornes, la couverture statistique d’un véritable intervalle de confiance ; cette plage mesure la sensibilité de l’estimation au rééchantillonnage des bureaux, non une probabilité de contenir la vraie valeur. Les reports extrêmes, nuls ou proches de 100 %, sont les plus fragiles, car la contrainte de positivité y joue à plein : un report affiché à 0 % n’est parfois qu’une estimation bruitée ramenée à la borne. C’est pourquoi les candidats dont le report semble le plus fort ou le plus faible ne sont mis en avant qu’une fois leur plage de stabilité vérifiée, faute de quoi l’on prendrait pour un comportement électoral ce qui relève de la régression vers la moyenne.

Il existe toutefois quelques circonscriptions où le $R^2$ est inférieur à 90 % :

In [14]:
r2_faible(df)

circonscription,destination,r2_echantillon
str,str,f64
"""69-08""","""ENS-HOR""",-0.048941
"""49-05""","""NFP""",0.082992
"""69-05""","""non exprimés (T2)""",0.138091
"""69-01""","""non exprimés (T2)""",0.669611
"""69-05""","""NFP""",0.718665
…,…,…
"""92-06""","""LR""",0.887385
"""988-02""","""non exprimés (T2)""",0.887734
"""971-02""","""RN""",0.893415


Parmi les circonscriptions concernées, on retrouve des situations de dissidence (69-06, 75-15, 93-07) et un poids lourd du gouvernement (78-10, Aurore Bergé) qui peuvent expliquer des comportements locaux moins homogènes. On écarte les reports correspondants par la suite.

Ce filtre sur le $R^2$ ne doit toutefois pas faire illusion : un mauvais ajustement signale une circonscription peu fiable, mais la réciproque est fausse. Une circonscription au $R^2$ élevé dont les listes de premier tour ont des soutiens quasi colinéaires d’un bureau à l’autre produit des coefficients tout aussi instables, sans que l’ajustement n’en avertisse. Ce sont le conditionnement de la matrice de premier tour et l’écart type bootstrap, et non le seul $R^2$, qui permettent d’écarter ces cas. De même, lorsqu’on agrège les reports de plusieurs circonscriptions pour conclure au niveau d’un bloc, il convient de pondérer chaque circonscription par sa fiabilité plutôt que de les traiter à égalité.